In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from transformers import AutoImageProcessor, AutoModelForImageClassification
from tqdm.auto import tqdm
from torchvision import transforms

#sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

DATA_DIR = Path("geo_dataset")  # change this
TRAIN_DIR = DATA_DIR / "train"
HOLDOUT_DIR = DATA_DIR / "holdout_public"
LABELS_PATH = DATA_DIR / "train_labels.csv"

/home/utn/poli22wo/miniconda3/envs/dl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
OUTPUT_DIR = Path("outputs/integrated_blend")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

best_model_path = OUTPUT_DIR / "best_model.pt"
checkpoint_path = OUTPUT_DIR / "training_checkpoint.pt"
history_path = OUTPUT_DIR / "history.csv"

In [3]:
df = pd.read_csv(LABELS_PATH)

print(df.shape)
display(df.head())

(11758, 5)


,filename,country,iso,lat,lng
0,1fcb4a43864244259b7d8f4a00f1e475.jpg,Turkey,TR,40.112290,38.304629
1,742f45b0211c44ffb19ad84931ea519c.jpg,France,FR,48.094103,-1.994316
2,152a13ef249d4efa95c51ed93f026284.jpg,Turkey,TR,41.324741,27.961821
3,81ce4a88bff14fef8420bca42019b12b.jpg,France,FR,47.585855,-2.971004
4,6fbcfe523e1349759e6060d632d52e54.jpg,United_Kingdom,GB,55.698094,-4.305315


In [4]:
countries = sorted(df["country"].unique())

country_to_index = {
    country: index
    for index, country in enumerate(countries)
}

index_to_country = {
    index: country
    for country, index in country_to_index.items()
}

df["country_index"] = df["country"].map(
    country_to_index
)

print(country_to_index)

{'Belarus': 0, 'Finland': 1, 'France': 2, 'Germany': 3, 'Iceland': 4, 'Italy': 5, 'Norway': 6, 'Poland': 7, 'Spain': 8, 'Sweden': 9, 'Turkey': 10, 'United_Kingdom': 11}


Validation Split

In [5]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["country"],
)

print("Training images:", len(train_df))
print("Validation images:", len(val_df))

Training images: 9406
Validation images: 2352


Model Verification

In [6]:
MODEL_NAME = "apple/mobilevitv2-1.0-imagenet1k-256"

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)

model = AutoModelForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=14,
    ignore_mismatched_sizes=True,
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print("Device:", device)

[transformers] You passed `num_labels=14` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 269/269 [00:00<00:00, 47013.12it/s]
[transformers] MobileViTV2ForImageClassification LOAD REPORT from: apple/mobilevitv2-1.0-imagenet1k-256
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 512]) vs model:torch.Size([14, 512])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([14])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Device: cuda


In [7]:
total_params = sum(p.numel() for p in model.parameters())

print(f"Parameters: {total_params:,}")
assert total_params <= 5_000_000

Parameters: 4,396,023


In [8]:
country_model_path = Path(
    best_model_path
)

best_weights = torch.load(
    best_model_path,
    map_location=device,
    weights_only=True,
)

model.load_state_dict(best_weights)
model.eval()

MobileViTV2ForImageClassification(
  (mobilevitv2): MobileViTV2Model(
    (conv_stem): MobileViTV2ConvLayer(
      (convolution): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (normalization): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (activation): SiLU()
    )
    (encoder): MobileViTV2Encoder(
      (layer): ModuleList(
        (0): MobileViTV2MobileNetLayer(
          (layer): ModuleList(
            (0): MobileViTV2InvertedResidual(
              (expand_1x1): MobileViTV2ConvLayer(
                (convolution): Conv2d(32, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
                (normalization): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
                (activation): SiLU()
              )
              (conv_3x3): MobileViTV2ConvLayer(
                (convolution): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), gr

In [9]:
checkpoint_path = Path(
    "outputs/integrated_blend/training_checkpoint.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False,
)

country_centres_tensor = checkpoint[
    "country_centres"
].to(device)

COORDINATE_WEIGHT = checkpoint[
    "coordinate_weight"
]

Image Processor and Dataset

In [10]:
class GeolocationDataset(Dataset):
    def __init__(
        self,
        dataframe,
        image_dir,
        processor,
        transform=None,
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.processor = processor
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image_path = self.image_dir / row["filename"]
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        pixel_values = self.processor(
            images=image,
            return_tensors="pt",
        )["pixel_values"].squeeze(0)

        coordinates = torch.tensor(
            [
                row["lat"] / 90,
                row["lng"] / 180,
            ],
            dtype=torch.float32,
        )

        country_index = torch.tensor(
            row["country_index"],
            dtype=torch.long,
        )

        return pixel_values, coordinates, country_index

In [11]:
train_dataset = GeolocationDataset(
    train_df,
    TRAIN_DIR,
    processor,
    transform=None,
)

val_dataset = GeolocationDataset(
    val_df,
    TRAIN_DIR,
    processor,
    transform=None,
)

In [12]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

#Loss and Optimizer

In [12]:
coordinate_loss_function = nn.MSELoss()
country_loss_function = nn.CrossEntropyLoss()

COUNTRY_LOSS_WEIGHT = 0.01

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-5,
)

In [13]:
images, coordinates, country_labels = next(iter(train_loader))

images = images.to(device)
coordinates = coordinates.to(device)
country_labels = country_labels.to(device)

print("Images:", images.shape)
print("Coordinates:", coordinates.shape)
print("Country Labels:", country_labels.shape)

Images: torch.Size([32, 3, 256, 256])
Coordinates: torch.Size([32, 2])
Country Labels: torch.Size([32])


In [13]:
def haversine_km(lat1, lng1, lat2, lng2):
    radius = 6371.0088

    lat1 = np.radians(lat1)
    lng1 = np.radians(lng1)
    lat2 = np.radians(lat2)
    lng2 = np.radians(lng2)

    difference = (
        np.sin((lat2 - lat1) / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin((lng2 - lng1) / 2) ** 2
    )

    return (
        2
        * radius
        * np.arcsin(
            np.sqrt(np.clip(difference, 0, 1))
        )
    )

Verifying old results

In [14]:
all_predictions = []
all_coordinates = []

with torch.no_grad():
    for images, coordinates, country_labels in tqdm(
        val_loader,
        desc="Verifying saved model",
    ):
        images = images.to(device)

        outputs = model(
            pixel_values=images
        ).logits

        direct_coordinates = torch.tanh(
            outputs[:, :2]
        )

        country_logits = outputs[:, 2:]

        country_probabilities = torch.softmax(
            country_logits,
            dim=1,
        )

        country_coordinates = (
            country_probabilities
            @ country_centres_tensor
        )

        final_coordinates = (
            COORDINATE_WEIGHT
            * direct_coordinates
            + (1 - COORDINATE_WEIGHT)
            * country_coordinates
        )

        all_predictions.append(
            final_coordinates.cpu().numpy()
        )

        all_coordinates.append(
            coordinates.numpy()
        )

Verifying saved model: 100%|██████████| 74/74 [00:11<00:00,  6.37it/s]


In [15]:
all_predictions = np.concatenate(
    all_predictions
)

all_coordinates = np.concatenate(
    all_coordinates
)

predictions_degrees = all_predictions.copy()
coordinates_degrees = all_coordinates.copy()

predictions_degrees[:, 0] *= 90
predictions_degrees[:, 1] *= 180

coordinates_degrees[:, 0] *= 90
coordinates_degrees[:, 1] *= 180

distances = haversine_km(
    coordinates_degrees[:, 0],
    coordinates_degrees[:, 1],
    predictions_degrees[:, 0],
    predictions_degrees[:, 1],
)

print(f"Mean: {np.mean(distances):.1f} km")
print(f"Median: {np.median(distances):.1f} km")
print(f"Within 200 km: {np.mean(distances < 200):.2%}")
print(f"Within 750 km: {np.mean(distances < 750):.2%}")

Mean: 614.3 km
Median: 373.1 km
Within 200 km: 25.17%
Within 750 km: 73.72%


Normalise country centres

In [15]:
country_centres = np.zeros(
    (len(countries), 2),
    dtype=np.float32,
)

for country, index in country_to_index.items():
    country_rows = train_df[
        train_df["country"] == country
    ]

    country_centres[index, 0] = (
        country_rows["lat"].median() / 90
    )

    country_centres[index, 1] = (
        country_rows["lng"].median() / 180
    )

In [18]:
COORDINATE_WEIGHT = 0.25

Converting centres to a gpu tensor

In [16]:
country_centres_tensor = torch.tensor(
    country_centres,
    dtype=torch.float32,
    device=device,
)

Full Train + Validation Loop (Current best-> Epochs: 20, median: 709) (Reload Optimizer before continuing training)

In [19]:
start_epoch = 0
END_EPOCH = 15

history = []

best_median = 397.493157
best_epoch = 0

epochs_without_improvement = 0
patience = 4

for epoch in range(start_epoch, END_EPOCH):

    # --------------------
    # Training
    # --------------------
    model.train()
    total_training_loss = 0

    training_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{END_EPOCH} - Training",
    )

    for images, coordinates, country_labels in training_bar:
        images = images.to(device)
        coordinates = coordinates.to(device)
        country_labels = country_labels.to(device)

        optimizer.zero_grad()

        outputs = model(
            pixel_values=images
        ).logits

        #Split the outputs
        direct_coordinates = torch.tanh(
            outputs[:, :2]
        )

        country_logits = outputs[:, 2:]

        country_probabilities = torch.softmax(
            country_logits,
            dim=1,
        )

        country_coordinates = (
            country_probabilities
            @ country_centres_tensor
        )

        final_coordinates = (
            COORDINATE_WEIGHT
            * direct_coordinates
            + (1 - COORDINATE_WEIGHT)
            * country_coordinates
        )

        coordinate_loss = coordinate_loss_function(
            final_coordinates,
            coordinates,
        )

        country_loss = country_loss_function(
            country_logits,
            country_labels,
        )

        loss = (
            coordinate_loss
            + COUNTRY_LOSS_WEIGHT * country_loss
        )

        loss.backward()
        optimizer.step()

        total_training_loss += loss.item()

        training_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    average_training_loss = (
        total_training_loss / len(train_loader)
    )

    # --------------------
    # Validation
    # --------------------
    model.eval()

    total_validation_loss = 0
    all_predictions = []
    all_coordinates = []

    validation_bar = tqdm(
        val_loader,
        desc=f"Epoch {epoch + 1}/{END_EPOCH} - Validation",
    )

    correct_country_predictions = 0
    number_of_validation_images = 0

    with torch.no_grad():
        for images, coordinates, country_labels in validation_bar:
            images = images.to(device)
            coordinates = coordinates.to(device)
            country_labels = country_labels.to(device)

            outputs = model(
                pixel_values=images
            ).logits

            # Direct coordinate prediction
            direct_coordinates = torch.tanh(
                outputs[:, :2]
            )

            # Country prediction
            country_logits = outputs[:, 2:]

            country_probabilities = torch.softmax(
                country_logits,
                dim=1,
            )

            # Probability-weighted country coordinate
            country_coordinates = (
                country_probabilities
                @ country_centres_tensor
            )

            # Final blended coordinate
            final_coordinates = (
                COORDINATE_WEIGHT
                * direct_coordinates
                + (1 - COORDINATE_WEIGHT)
                * country_coordinates
            )

            # Evaluate the same blended coordinate used during training
            coordinate_loss = coordinate_loss_function(
                final_coordinates,
                coordinates,
            )

            country_loss = country_loss_function(
                country_logits,
                country_labels,
            )

            loss = (
                coordinate_loss
                + COUNTRY_LOSS_WEIGHT * country_loss
            )

            total_validation_loss += loss.item()

            # Store the final blended prediction, not the direct prediction
            all_predictions.append(
                final_coordinates.cpu().numpy()
            )

            all_coordinates.append(
                coordinates.cpu().numpy()
            )

            predicted_countries = country_logits.argmax(
                dim=1
            )

            correct_country_predictions += (
                predicted_countries == country_labels
            ).sum().item()

            number_of_validation_images += (
                country_labels.size(0)
            )

        country_accuracy = (
            correct_country_predictions
            / number_of_validation_images
        )       

    average_validation_loss = (
        total_validation_loss / len(val_loader)
    )

    # Combine validation batches
    all_predictions = np.concatenate(all_predictions)
    all_coordinates = np.concatenate(all_coordinates)

    # Convert normalized coordinates back into degrees
    predictions_degrees = all_predictions.copy()
    coordinates_degrees = all_coordinates.copy()

    predictions_degrees[:, 0] *= 90
    predictions_degrees[:, 1] *= 180

    coordinates_degrees[:, 0] *= 90
    coordinates_degrees[:, 1] *= 180

    # Calculate geographic distances
    distances = haversine_km(
        coordinates_degrees[:, 0],
        coordinates_degrees[:, 1],
        predictions_degrees[:, 0],
        predictions_degrees[:, 1],
    )

    mean_distance = np.mean(distances)
    median_distance = np.median(distances)
    within_200 = np.mean(distances < 200)
    within_750 = np.mean(distances < 750)

    # Save this epoch's results
    history.append({
        "epoch": epoch + 1,
        "training_loss": average_training_loss,
        "validation_loss": average_validation_loss,
        "mean_km": mean_distance,
        "median_km": median_distance,
        "within_200": within_200,
        "within_750": within_750,
        "country_accuracy": country_accuracy
    })

    # Display this epoch's results
    print(f"\nEpoch {epoch + 1} results")
    print(f"Training loss: {average_training_loss:.4f}")
    print(f"Validation loss: {average_validation_loss:.4f}")
    print(f"Mean distance: {mean_distance:.1f} km")
    print(f"Median distance: {median_distance:.1f} km")
    print(f"Within 200 km: {within_200:.2%}")
    print(f"Within 750 km: {within_750:.2%}")
    print(f"Country accuracy: {country_accuracy:.2%}")

    if median_distance < best_median:
        best_median = median_distance
        best_epoch = epoch + 1

        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            best_model_path,
        )

        print("Saved new best model.")

    else:
        epochs_without_improvement += 1

        print(
            "Epochs without improvement:",
            epochs_without_improvement,
        )

    if epochs_without_improvement >= patience:
        print("Early stopping.")
        break

Epoch 1/15 - Validation: 100%|██████████| 74/74 [00:11<00:00,  6.41it/s]



Epoch 1 results
Training loss: 0.0005
Validation loss: 0.0212
Mean distance: 623.2 km
Median distance: 386.2 km
Within 200 km: 23.64%
Within 750 km: 73.89%
Country accuracy: 66.24%
Saved new best model.


Epoch 2/15 - Validation: 100%|██████████| 74/74 [00:11<00:00,  6.66it/s]



Epoch 2 results
Training loss: 0.0004
Validation loss: 0.0218
Mean distance: 614.3 km
Median distance: 373.1 km
Within 200 km: 25.17%
Within 750 km: 73.72%
Country accuracy: 65.90%
Saved new best model.


Epoch 3/15 - Validation: 100%|██████████| 74/74 [00:11<00:00,  6.41it/s]


...


Epoch 4 results
Training loss: 0.0004
Validation loss: 0.0211
Mean distance: 618.1 km
Median distance: 373.4 km
Within 200 km: 25.09%
Within 750 km: 73.60%
Country accuracy: 65.69%
Epochs without improvement: 2


Epoch 5/15 - Validation: 100%|██████████| 74/74 [00:11<00:00,  6.51it/s]



Epoch 5 results
Training loss: 0.0004
Validation loss: 0.0215
Mean distance: 608.3 km
Median distance: 374.0 km
Within 200 km: 25.30%
Within 750 km: 73.81%
Country accuracy: 66.03%
Epochs without improvement: 3


Epoch 6/15 - Validation: 100%|██████████| 74/74 [00:11<00:00,  6.44it/s]


Epoch 6 results
Training loss: 0.0003
Validation loss: 0.0215
Mean distance: 624.5 km
Median distance: 399.1 km
Within 200 km: 22.70%
Within 750 km: 73.17%
Country accuracy: 66.16%
Epochs without improvement: 4
Early stopping.


In [21]:
torch.save(
    {
        # Existing training information
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_median": best_median,
        "history": history,
        "best_epoch": best_epoch,
        "patience": patience,
        "epochs_without_improvement": (
            epochs_without_improvement
        ),

        # New model information
        "model_name": MODEL_NAME,
        "num_labels": 14,
        "parameter_count": total_params,

        # New country-blending information
        "country_centres": (
            country_centres_tensor
            .detach()
            .cpu()
        ),
        "country_to_index": country_to_index,
        "index_to_country": index_to_country,
        "coordinate_weight": COORDINATE_WEIGHT,
        "country_loss_weight": (
            COUNTRY_LOSS_WEIGHT
        ),
    },
    checkpoint_path,
)

In [ ]:
history_df = pd.DataFrame(history)

history_df.to_csv(
    history_path,
    index=False,
)